In [1]:
import sys
import networkx as nx 
import numpy as np
import numpy.linalg as la 
import math 
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
import plotly
import pickle
import pandas as pd
from scipy.stats import gaussian_kde


import plotly.io as pio
# pio.orca.config.use_xvfb = True

import seaborn as sns
sns.set_theme(style = 'darkgrid')
sns.set_palette("crest")
crest_palette = sns.color_palette("crest", as_cmap=True).colors

from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy.stats import gaussian_kde

sys.path.append('/home/csubrama/projects/conda/envs/')

import kirchhoff
import kirchhoff.init_random as kir
import kirchhoff.circuit_dual as kcd
import kirchhoff.circuit_flow as kfc
import kirchhoff.init_dual as kid
import kirchhoff.init_crystal as kic
import kirchhoff.draw_networkx as kdn

import snarlpy
import snarlpy.tangledGenerators as tg 
import snarlpy.edgePriority as spe

import warnings 
warnings.filterwarnings("ignore", category = DeprecationWarning)
warnings.filterwarnings("ignore", category = np.VisibleDeprecationWarning)

In [2]:
pio.renderers.default = "notebook_connected"

In [3]:
def create_graph(vertices):
    G = nx.Graph()
    for i, coord in enumerate(vertices):
        G.add_node(i, pos=coord)
        
    list_n = list(G.nodes())
    dict_d = {}
    threshold = 1.
    for idx_n, n in enumerate(list_n[:-1]):
        for m in list_n[idx_n+1:]:
            p = np.array(G.nodes[n]['pos'])
            q = np.array(G.nodes[m]['pos'])
            dict_d[(n, m)] = round(np.linalg.norm(p-q),5)

    for nm in dict_d:
        if dict_d[nm] <= threshold:
            G.add_edge(*nm)
            
#     kg = kfc.setup_default_flow_circuit(G)
#     print(nx.info(kg))
#     kg.set_source_landscape()
#     kg.set_plexus_landscape()
#     fig = kg.plot_circuit()
#     fig.show()
    
    return G

In [4]:
def dualGraph(vertices):
    G1 = create_graph(vertices[0])
    G2 = create_graph(vertices[1])

    for n in G1.nodes():
        p = np.array(G1.nodes[n]['pos'])
        # display(p.shape)
        p1 = np.dot(p, rot_mat.T)
        p2 = np.add(coords_offset, p1)

#         G1.nodes[n]['pos'] = self.lattice_constant*p2
        G1.nodes[n]['pos'] = np.array(G1.nodes[n]['pos'])
        # display(G1.nodes())

    graph_sets = [G1, G2]
    
    return graph_sets

In [5]:
class NetworkxDualGraphs(kid.NetworkxDual):
    def __init__(self, vertices):
        super(NetworkxDualGraphs, self).__init__()
        self.lattice_constant = 1
        self.translation_length = 1
        self.dualGraph(vertices)

    def dualGraph(self, vertices):
        G1 = create_graph(vertices[0])
        G2 = create_graph(vertices[1])

#         for n in G1.nodes():
#             p = np.array(G1.nodes[n]['pos'])
#             # display(p.shape)
#             p1 = np.dot(p, rot_mat.T)
#             p2 = np.add(coords_offset, p1)
            
#             G1.nodes[n]['pos'] = self.lattice_constant*p2
#             G1.nodes[n]['pos'] = np.array(G1.nodes[n]['pos'])
#             # display(G1.nodes())

        self.layer = [G1, G2]

In [6]:
def init_dualGraph(dual_type, vertices):
    # kid.init_dualCatenation
    graphSet = NetworkxDualGraphs(vertices)

    # kcd.initialize_dual_from_catenation
    circuitSet = kcd.construct_from_graphSet(graphSet, 
                                             circuit_type = 'default')
    for i,g in enumerate(graphSet.layer):
        circuitSet[i].info = circuitSet[i].set_info(g, dual_type)

    kirchhoff_dual = kcd.DualCircuit(circuitSet)
    
    return kirchhoff_dual

In [7]:
bouncerList = []

def set_print_labels(D):
    
    for i, khf in enumerate(D.layer):

        khf.nodes['label'] = [n for n in khf.G.nodes()]
        khf.edges['label'] = [e for i, e in enumerate(khf.G.edges())]  
    
    return D

def createLabelGraphs(vertices):
    D = init_dualGraph(dual_type = NetworkxDualGraphs, 
                             vertices = vertices)
    D = set_print_labels(D)
    # display(D)
    return D

In [8]:
def update_plot_eigvecs(D, operator, eigvec_idx):
    colors = []
    eigvecs_indexed = []
    
    U, singvals, V = np.linalg.svd(operator)
    singvals[abs(singvals.real) <= 1e-10] = 0
#     singvals = np.round(singvals, 4)
    print(singvals)
    idx_sorted_singvals = np.argsort(singvals.real)[::-1]
    
    # Get the indices of the nonzero singular values
    nonzero_indices = np.where(abs(singvals.real) > 1e-10)[0]
    
    # Get the nonzero left and right singular vectors
    eigvecs = (eigvecs_left, eigvecs_right) = (U.real, V.T.real)
        
        # Get the indexed eigenvalue
    idx_singval = idx_sorted_singvals[eigvec_idx]
    eigvecs_indexed.append(eigvecs[0].real[:, idx_singval])
    
    for i, k in enumerate(D.layer):
        k.edges['priority'] = abs(eigvecs[i][:,eigvec_idx])
        colors.append(k.edges['priority'])
    
    kwargs = {
        'color_nodes': ['#030512','#030512'],
        'color_edges': colors,
        'colorbar': {'show': True, 'label': 'Eigenvector magnitude (|component|)'},
        'colormap': ['BuGn','RdPu'],
        'axis': True
    }
    
    fig = D.plot_circuit('priority', **kwargs)   
    
    return fig

In [28]:
vertex_coordinates_1 = np.array([
                        (0,0,0), 
                        (0,1/np.sqrt(2),0), 
                        (1/np.sqrt(2),0,1/np.sqrt(2)), 
                        (1/np.sqrt(2),1/np.sqrt(2),1/np.sqrt(2)), 
                        (1/np.sqrt(2),0,-1/np.sqrt(2)), 
                        (1/np.sqrt(2),1/np.sqrt(2),-1/np.sqrt(2)) #, 
#                         (0,1/np.sqrt(2),2/np.sqrt(2)), 
#                         (0,0,2/np.sqrt(2)) , 
#                         (1/np.sqrt(2),1/np.sqrt(2),3/np.sqrt(2)), 
#                         (1/np.sqrt(2),0,3/np.sqrt(2))
                    ])

vertex_coordinates_2 = np.array([
                        (0.5/np.sqrt(2),-0.5/np.sqrt(2),0), 
                        (0.5/np.sqrt(2),0.5/np.sqrt(2),0), 
                        (-0.5/np.sqrt(2),-0.5/np.sqrt(2),1/np.sqrt(2)), 
                        (-0.5/np.sqrt(2),0.5/np.sqrt(2),1/np.sqrt(2)), 
                        (-0.5/np.sqrt(2),-0.5/np.sqrt(2),-1/np.sqrt(2)), 
                        (-0.5/np.sqrt(2),0.5/np.sqrt(2),-1/np.sqrt(2)) #, 
#                         (0.5/np.sqrt(2),0.5/np.sqrt(2),2/np.sqrt(2)), 
#                         (0.5/np.sqrt(2),-0.5/np.sqrt(2), 2/np.sqrt(2)) , 
#                         (-0.5/np.sqrt(2),-0.5/np.sqrt(2),3/np.sqrt(2)), 
#                         (-0.5/np.sqrt(2),0.5/np.sqrt(2),3/np.sqrt(2))
                    ])

vertices = [vertex_coordinates_1, vertex_coordinates_2]

D = createLabelGraphs(vertices)
graph_sets = [k.G for k in D.layer]
p, lk_mat, graph_matrices, cycles = spe.getEdgeLinkageOperator(graph_sets)

fig = D.plot_circuit('')
fig.show()

operator = p 
eigvec_idx = 1

fig = update_plot_eigvecs(D, operator, eigvec_idx)
fig.show()

[0.25819889 0.25819889 0.         0.         0.         0.
 0.        ]
